# GPU memory check v2 for PEFTGuard retraining (no real data needed)

v1 crashed the kernel on an A100 with `Error invalid argument at line 177 in file /src/csrc/ops.cu` (a bitsandbytes CUDA error, which cannot be caught in Python).
**v2 runs every plan in its own subprocess**, so a crash is reported as CRASHED and the next plan still runs.

It answers: (1) which optimizer/precision plans fit this GPU, with peak memory and seconds per sample; (2) whether the crash is the 8-bit Adam optimizer
choking on the single 2^31-element weight (`fc1`), using two probes (2^30 vs 2^31 elements).

Use an **A100**. Runtime > Run all takes about 10-20 minutes. Copy the printed table back to the team.

In [ ]:
!pip -q install safetensors bitsandbytes
!rm -rf /content/repo && git clone --depth 1 https://github.com/JuliaIssa2/FYP-AI-Antivirus.git /content/repo
!git -C /content/repo log -1 --format='%h %s'
import torch, bitsandbytes
p = torch.cuda.get_device_properties(0)
print("GPU:", p.name, f"{p.total_memory/2**30:.1f} GB | torch", torch.__version__, "| bitsandbytes", bitsandbytes.__version__, "| CUDA", torch.version.cuda)
print("torch.optim.Adafactor available:", hasattr(torch.optim, "Adafactor"))

In [ ]:
%%writefile /content/trial.py
import contextlib, gc, json, os, sys, time
sys.path.insert(0, os.environ.get("TRIAL_REPO", "/content/repo"))
import numpy as np
import torch
import torch.nn.functional as F

plan = sys.argv[1]
DEVICE = os.environ.get("TRIAL_DEVICE", "cuda")
DIM, LAYERS = int(os.environ.get("TRIAL_DIM", 4096)), int(os.environ.get("TRIAL_LAYERS", 32))
CUDA = DEVICE == "cuda"
sync = torch.cuda.synchronize if CUDA else (lambda: None)
peak = (lambda: torch.cuda.max_memory_allocated() / 2**30) if CUDA else (lambda: 0.0)
def emit(d): print("RESULT " + json.dumps(d), flush=True)

if plan == "selftest_crash":
    os._exit(7)                                    # simulates a hard CUDA abort
try:
    if plan.startswith("probe_"):                  # one parameter of 2**k elements, one 8-bit Adam step
        import bitsandbytes as bnb
        n = 2 ** int(plan.split("_")[1])
        p = torch.nn.Parameter(torch.empty(n, device=DEVICE).normal_(0, 0.01))
        p.grad = torch.empty(n, device=DEVICE).normal_(0, 0.01)
        bnb.optim.Adam8bit([p], lr=1e-4).step(); sync()
        emit({"ok": True, "peak_gb": peak(), "detail": f"one parameter of {n} elements"})
        sys.exit(0)

    from shared_protocol.detectors.peftguard.model import PGLlama2
    from shared_protocol.detectors.peftguard.converter import build_input
    optname, pdtype, autocast = {"A": ("adam8bit", torch.float32, True), "B": ("adam", torch.float32, False),
                                 "C": ("adam", torch.bfloat16, False), "D": ("adafactor", torch.float32, False)}[plan]
    rng = np.random.default_rng(0)
    factors = {(l, m): (rng.normal(0, 0.02, (16, DIM)).astype(np.float32), rng.normal(0, 0.02, (DIM, 16)).astype(np.float32))
               for l in range(LAYERS) for m in ("q_proj", "v_proj")}
    with torch.device(DEVICE):
        model = PGLlama2(DIM, LAYERS)
    if pdtype != torch.float32:
        model = model.to(pdtype)
    if optname == "adam8bit":
        import bitsandbytes as bnb
        opt = bnb.optim.Adam8bit(model.parameters(), lr=1e-4, weight_decay=1e-5)
    elif optname == "adam":
        opt = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    else:
        opt = torch.optim.Adafactor(model.parameters(), lr=1e-4, weight_decay=1e-5)
    target = F.one_hot(torch.tensor([1], device=DEVICE), 2).float()
    times, loss_val, accum = [], None, 2
    for _ in range(3):                             # 3 optimizer steps, each accumulating 2 samples
        opt.zero_grad(set_to_none=True)
        for _ in range(accum):
            t0 = time.time()
            x = build_input(factors, dim=DIM, layers=LAYERS, device=DEVICE, dtype=pdtype)
            ctx = torch.autocast("cuda", dtype=torch.bfloat16) if (autocast and CUDA) else contextlib.nullcontext()
            with ctx:
                out = model(x).float()
            loss = F.binary_cross_entropy_with_logits(out, target)
            (loss / accum).backward()
            sync(); times.append(time.time() - t0); loss_val = float(loss.detach())
            del x, out, loss
        opt.step()
    sync()
    emit({"ok": True, "peak_gb": peak(), "sec_per_sample": float(np.median(times[1:])), "loss_finite": bool(np.isfinite(loss_val))})
except torch.cuda.OutOfMemoryError:
    emit({"ok": False, "reason": "OUT OF MEMORY", "peak_gb": peak()})
except Exception as e:
    emit({"ok": False, "reason": "ERROR: " + repr(e)[:300]})

In [ ]:
import json, subprocess, sys, time
def run(plan, timeout=1500):
    t0 = time.time()
    p = subprocess.run([sys.executable, "/content/trial.py", plan], capture_output=True, text=True, timeout=timeout)
    line = next((l for l in reversed(p.stdout.splitlines()) if l.startswith("RESULT ")), None)
    if line:
        r = json.loads(line[7:])
    else:   # the process died without reporting: a hard crash (e.g. a CUDA abort)
        r = {"ok": False, "reason": f"CRASHED (exit code {p.returncode})", "tail": (p.stdout + p.stderr).strip().splitlines()[-3:]}
    r["wall_s"] = round(time.time() - t0)
    return r

LABELS = {"probe_30": "probe: 8-bit Adam, ONE tensor of 2^30 elements", "probe_31": "probe: 8-bit Adam, ONE tensor of 2^31 elements (= fc1)",
          "D": "D  fp32 weights, Adafactor (faithful forward/backward)", "C": "C  bf16 weights, Adam (all in bf16)",
          "B": "B  fp32 weights, fp32 Adam (the authors' recipe)", "A": "A  fp32 weights, 8-bit Adam + bf16 autocast"}
results = {}

## Run everything (each plan in its own process, so a crash cannot stop the others)

In [ ]:
for plan in ["probe_30", "probe_31", "D", "C", "B", "A"]:
    results[plan] = run(plan)
    print(LABELS[plan], "->", {k: (round(v, 2) if isinstance(v, float) else v) for k, v in results[plan].items()})

## Summary to send to the team

In [ ]:
print("GPU:", p.name, f"{p.total_memory/2**30:.0f} GB | torch", torch.__version__, "| bitsandbytes", bitsandbytes.__version__)
for plan, r in results.items():
    if r.get("ok"):
        extra = f"{r['sec_per_sample']:.1f} s/sample, finite loss {r['loss_finite']}" if "sec_per_sample" in r else r.get("detail", "")
        print(f"  OK       {LABELS[plan]:60s} peak {r['peak_gb']:.1f} GB  {extra}")
    else:
        print(f"  FAILED   {LABELS[plan]:60s} {r.get('reason')}" + (f"  tail: {r['tail']}" if r.get("tail") else ""))
ok = [pl for pl in ("A", "B", "C", "D") if results[pl].get("ok")]
print("Plans that fit:", ok or "none")